# Data Leakage Check — Retail Revenue Forecasting

**Mục đích:** Kiểm tra rò rỉ dữ liệu (data leakage) theo thời gian giữa các tập train/validation/test trước khi đưa vào modeling.

**Phạm vi kiểm tra:**
- Khoảng giao thời gian giữa train (2019–2022 / 2019–2023), val (2023 / 2024), test (2025)
- Tính liên tục và toàn vẹn của chuỗi thời gian (gap, duplicate, sort)
- Missing values ở target (`revenue`)
- Phân bố các đặc trưng liên quan giai đoạn COVID



In [27]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "data").exists():
    raise FileNotFoundError("Could not locate a 'data' directory above the current path.")

print(ROOT)


d:\project_git\retail-revenue-forecasting


In [28]:
DATE_COLS = ["week_start"]

files = {
    "train_1": "train_2019_2022.csv",
    "val_1":   "validation_2023.csv",
    "train_2": "train_2019_2023.csv",
    "val_2":   "validation_2024.csv",
    "test":    "test_2025.csv",
    "df":      "cleaned_transactions.csv",
}

data = {
    name: pd.read_csv(ROOT / "data/processed" / fname, parse_dates=DATE_COLS)
    for name, fname in files.items()
}

train_1, val_1, train_2, val_2, test, df = (
    data["train_1"], data["val_1"], data["train_2"],
    data["val_2"], data["test"], data["df"],
)

In [29]:
df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
df["split_year_original"] = df["split_year"]
df["split_year"] = df["week_start"].dt.year
train = df[df["split_year"] <= 2022]
val_2023 = df[df["split_year"] == 2023]
val_2024 = df[df["split_year"] == 2024]
test = df[df["split_year"] == 2025]

print("Train:", len(train), "weeks")
print("Validation 2023:", len(val_2023), "weeks")
print("Validation 2024:", len(val_2024), "weeks")
print("Test 2025:", len(test), "weeks")

print("\nDate ranges:")

for name, data in [
    ("Train", train),
    ("Validation 2023", val_2023),
    ("Validation 2024", val_2024),
    ("Test", test),
]:
    if len(data) > 0:
        print(
            f"{name:18s}: "
            f"{data['week_start'].min().date()} "
            f"→ "
            f"{data['week_start'].max().date()}"
        )

Train: 6816 weeks
Validation 2023: 1756 weeks
Validation 2024: 1728 weeks
Test 2025: 1700 weeks

Date ranges:
Train             : 2018-12-31 → 2022-12-26
Validation 2023   : 2023-01-02 → 2023-12-25
Validation 2024   : 2024-01-01 → 2024-12-30
Test              : 2025-01-06 → 2025-12-29


In [30]:
#train(29-22) ∩ val 23
overlap_train_val23 = set(train_1["week_start"]) & set(val_1["week_start"])
#train(29-23) ∩ val 24
overlap_train_val24 = set(train_2["week_start"]) & set(val_2["week_start"])
#val 24 ∩ test 25

overlap_val_test    = set(val_2["week_start"]) & set(test["week_start"])

assert not overlap_train_val23, f"Overlap: {overlap_train_val23}"
assert not overlap_train_val24, f"Overlap: {overlap_train_val24}"
assert not overlap_val_test, f"Overlap: {overlap_val_test}"

In [31]:

print("Missing week_start:", df["week_start"].isna().sum())
print("Min date:", df["week_start"].min())
print("Max date:", df["week_start"].max())

Missing week_start: 0
Min date: 2018-12-31 00:00:00
Max date: 2025-12-29 00:00:00


In [32]:
duplicate_count = df["week_start"].duplicated().sum()

print("Duplicate weeks:", duplicate_count)

Duplicate weeks: 11634


In [33]:
print("Duplicate full rows:", df.duplicated().sum())

Duplicate full rows: 0


In [34]:
is_sorted = df["week_start"].is_monotonic_increasing

print("Chronological:", is_sorted)


Chronological: True


In [35]:
week_diff = df["week_start"].diff().dt.days.dropna()

abnormal_gaps = week_diff[week_diff != 7]

print("Total week gaps checked:", len(week_diff))
print("Abnormal gaps:", len(abnormal_gaps))

if len(abnormal_gaps) == 0:
    print("All weeks are continuous")
else:
    print("Abnormal week gaps detected")

    print("\nExamples:")
    print(
        df.loc[
            abnormal_gaps.index,
            ["week_start"]
        ].head(10)
    )

Total week gaps checked: 11999
Abnormal gaps: 11634
Abnormal week gaps detected

Examples:
   week_start
1  2018-12-31
2  2018-12-31
3  2018-12-31
4  2018-12-31
5  2018-12-31
6  2018-12-31
7  2018-12-31
8  2018-12-31
9  2018-12-31
10 2018-12-31


In [36]:
print("Missing revenue:", df["revenue"].isna().sum())

Missing revenue: 0


In [37]:
covid_columns = [
        "is_covid_period",
        "is_covid_shock",
        "covid_phase"
    ]

for col in covid_columns:
    if col in df.columns:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))


is_covid_period
is_covid_period
False    8831
True     3169
Name: count, dtype: int64

is_covid_shock
is_covid_shock
False    11390
True       610
Name: count, dtype: int64

covid_phase
covid_phase
post_covid          6847
covid_adaptation    2559
pre_covid           1984
covid_shock          610
Name: count, dtype: int64


# Feature Engineering Leakage Check
**Phạm vi kiểm tra:**
- Toàn bộ lag feature có đúng `shift(k)`;
- Rolling chỉ sử dụng dữ liệu từ `t-1` trở về trước;
- Outlier flag sử dụng `lag_1` và ngưỡng p90 lịch sử theo fold;
- Ratio feature chỉ được tạo từ lag;
- Ranh giới Val2023, Val2024 và Test2025;
- kiểm tra `target_revenue` không bị đưa trực tiếp vào feature set.

In [38]:
FEATURE_DIR = ROOT / "data" / "features"
SERIES = ["total", "online", "store", "mobile"]
WINDOW_LIST = [4, 8, 12, 26]
LAG_LIST = [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 26, 52]
def load_feature_matrix(series_name):
    path = FEATURE_DIR / f"features_{series_name}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Not found: {path}. ")

    df_feat = pd.read_csv(path,parse_dates=["week_start"])
    df_feat = (df_feat.set_index("week_start").sort_index())

    return df_feat


feature_dfs = {
    name: load_feature_matrix(name)
    for name in SERIES
}

for name, df_feat in feature_dfs.items():
    print(
        f"{name.upper():7s}: "
        f"{len(df_feat)} rows x {len(df_feat.columns)} columns"
    )

TOTAL  : 366 rows x 44 columns
ONLINE : 366 rows x 44 columns
STORE  : 366 rows x 44 columns
MOBILE : 366 rows x 44 columns


In [39]:
target_check = []

for name, df_feat in feature_dfs.items():

    feature_cols = [
        c for c in df_feat.columns
        if c not in ["target_revenue", "split_year"]
    ]

    target_check.append({
        "series": name,
        "raw_revenue_absent": "revenue" not in df_feat.columns,
        "target_excluded_from_X": "target_revenue" not in feature_cols,
        "split_year_excluded_from_X": "split_year" not in feature_cols,
        "n_features": len(feature_cols),
    })

target_check_df = pd.DataFrame(target_check)
display(target_check_df)


,series,raw_revenue_absent,target_excluded_from_X,split_year_excluded_from_X,n_features
0,total,True,True,True,42
1,online,True,True,True,42
2,store,True,True,True,42
3,mobile,True,True,True,42


In [40]:
def check_lag_features(df_feat):
    target = df_feat["target_revenue"]
    rows = []

    for k in LAG_LIST:
        col = f"lag_revenue_{k}"

        if col not in df_feat.columns:
            rows.append({
                "feature": col,
                "expected": f"target.shift({k})",
                "passed": False,
            })
            continue

        expected = target.shift(k)

        passed = np.allclose(
            df_feat[col].to_numpy(dtype=float),
            expected.to_numpy(dtype=float),
            equal_nan=True,
            rtol=1e-10,
            atol=1e-10,
        )

        rows.append({
            "feature": col,
            "expected": f"target.shift({k})",
            "passed": passed,
        })

    return pd.DataFrame(rows)


lag_summary = []

for name, df_feat in feature_dfs.items():
    result = check_lag_features(df_feat)

    lag_summary.append({
        "series": name,
        "passed": bool(result["passed"].all()),
        "passed_features": int(result["passed"].sum()),
        "total_features": len(result),
    })

    print(f"\n{name.upper()}")
    display(result)

lag_summary_df = pd.DataFrame(lag_summary)
print("\nLag summary")
display(lag_summary_df)



TOTAL


,feature,expected,passed
0,lag_revenue_1,target.shift(1),True
1,lag_revenue_2,target.shift(2),True
2,lag_revenue_3,target.shift(3),True
3,lag_revenue_4,target.shift(4),True
4,lag_revenue_5,target.shift(5),True
5,lag_revenue_6,target.shift(6),True
6,lag_revenue_7,target.shift(7),True
7,lag_revenue_8,target.shift(8),True
8,lag_revenue_9,target.shift(9),True
9,lag_revenue_10,target.shift(10),True



ONLINE


,feature,expected,passed
0,lag_revenue_1,target.shift(1),True
1,lag_revenue_2,target.shift(2),True
2,lag_revenue_3,target.shift(3),True
3,lag_revenue_4,target.shift(4),True
4,lag_revenue_5,target.shift(5),True
5,lag_revenue_6,target.shift(6),True
6,lag_revenue_7,target.shift(7),True
7,lag_revenue_8,target.shift(8),True
8,lag_revenue_9,target.shift(9),True
9,lag_revenue_10,target.shift(10),True



STORE


,feature,expected,passed
0,lag_revenue_1,target.shift(1),True
1,lag_revenue_2,target.shift(2),True
2,lag_revenue_3,target.shift(3),True
3,lag_revenue_4,target.shift(4),True
4,lag_revenue_5,target.shift(5),True
5,lag_revenue_6,target.shift(6),True
6,lag_revenue_7,target.shift(7),True
7,lag_revenue_8,target.shift(8),True
8,lag_revenue_9,target.shift(9),True
9,lag_revenue_10,target.shift(10),True



MOBILE


,feature,expected,passed
0,lag_revenue_1,target.shift(1),True
1,lag_revenue_2,target.shift(2),True
2,lag_revenue_3,target.shift(3),True
3,lag_revenue_4,target.shift(4),True
4,lag_revenue_5,target.shift(5),True
5,lag_revenue_6,target.shift(6),True
6,lag_revenue_7,target.shift(7),True
7,lag_revenue_8,target.shift(8),True
8,lag_revenue_9,target.shift(9),True
9,lag_revenue_10,target.shift(10),True



Lag summary


,series,passed,passed_features,total_features
0,total,True,14,14
1,online,True,14,14
2,store,True,14,14
3,mobile,True,14,14


In [41]:
def check_rolling_features(df_feat):
    shifted = df_feat["target_revenue"].shift(1)
    rows = []

    for w in WINDOW_LIST:
        rolled = shifted.rolling(
            window=w,
            min_periods=w
        )

        expected_map = {
            f"rolling_mean_{w}": rolled.mean(),
            f"rolling_std_{w}": rolled.std(),
            f"rolling_min_{w}": rolled.min(),
            f"rolling_max_{w}": rolled.max(),
        }

        for col, expected in expected_map.items():

            if col not in df_feat.columns:
                rows.append({
                    "feature": col,
                    "expected": "shift(1) -> rolling",
                    "passed": False,
                })
                continue

            passed = np.allclose(
                df_feat[col].to_numpy(dtype=float),
                expected.to_numpy(dtype=float),
                equal_nan=True,
                rtol=1e-10,
                atol=1e-10,
            )

            rows.append({
                "feature": col,
                "expected": "shift(1) -> rolling",
                "passed": passed,
            })

    return pd.DataFrame(rows)


rolling_summary = []

for name, df_feat in feature_dfs.items():
    result = check_rolling_features(df_feat)

    rolling_summary.append({
        "series": name,
        "passed": bool(result["passed"].all()),
        "passed_features": int(result["passed"].sum()),
        "total_features": len(result),
    })

    print(f"\n{name.upper()}")
    display(result)

rolling_summary_df = pd.DataFrame(rolling_summary)
print("\nRolling summary")
display(rolling_summary_df)




TOTAL


,feature,expected,passed
0,rolling_mean_4,shift(1) -> rolling,True
1,rolling_std_4,shift(1) -> rolling,True
2,rolling_min_4,shift(1) -> rolling,True
3,rolling_max_4,shift(1) -> rolling,True
4,rolling_mean_8,shift(1) -> rolling,True
5,rolling_std_8,shift(1) -> rolling,True
6,rolling_min_8,shift(1) -> rolling,True
7,rolling_max_8,shift(1) -> rolling,True
8,rolling_mean_12,shift(1) -> rolling,True
9,rolling_std_12,shift(1) -> rolling,True



ONLINE


,feature,expected,passed
0,rolling_mean_4,shift(1) -> rolling,True
1,rolling_std_4,shift(1) -> rolling,True
2,rolling_min_4,shift(1) -> rolling,True
3,rolling_max_4,shift(1) -> rolling,True
4,rolling_mean_8,shift(1) -> rolling,True
5,rolling_std_8,shift(1) -> rolling,True
6,rolling_min_8,shift(1) -> rolling,True
7,rolling_max_8,shift(1) -> rolling,True
8,rolling_mean_12,shift(1) -> rolling,True
9,rolling_std_12,shift(1) -> rolling,True



STORE


,feature,expected,passed
0,rolling_mean_4,shift(1) -> rolling,True
1,rolling_std_4,shift(1) -> rolling,True
2,rolling_min_4,shift(1) -> rolling,True
3,rolling_max_4,shift(1) -> rolling,True
4,rolling_mean_8,shift(1) -> rolling,True
5,rolling_std_8,shift(1) -> rolling,True
6,rolling_min_8,shift(1) -> rolling,True
7,rolling_max_8,shift(1) -> rolling,True
8,rolling_mean_12,shift(1) -> rolling,True
9,rolling_std_12,shift(1) -> rolling,True



MOBILE


,feature,expected,passed
0,rolling_mean_4,shift(1) -> rolling,True
1,rolling_std_4,shift(1) -> rolling,True
2,rolling_min_4,shift(1) -> rolling,True
3,rolling_max_4,shift(1) -> rolling,True
4,rolling_mean_8,shift(1) -> rolling,True
5,rolling_std_8,shift(1) -> rolling,True
6,rolling_min_8,shift(1) -> rolling,True
7,rolling_max_8,shift(1) -> rolling,True
8,rolling_mean_12,shift(1) -> rolling,True
9,rolling_std_12,shift(1) -> rolling,True



Rolling summary


,series,passed,passed_features,total_features
0,total,True,16,16
1,online,True,16,16
2,store,True,16,16
3,mobile,True,16,16


In [42]:
def check_outlier_flag(df_feat):
    target = df_feat["target_revenue"]

    threshold = pd.Series(np.nan, index=df_feat.index, dtype=float)

    # Train 2019-2022
    train_mask = df_feat["split_year"] <= 2022
    p90_train = float(target[train_mask].quantile(0.90))
    threshold.loc[train_mask] = p90_train

    thresholds = {
        "train_2019_2022": p90_train
    }

    # Expanding historical threshold
    for year in [2023, 2024, 2025]:
        current_mask = df_feat["split_year"] == year

        if current_mask.any():
            history_mask = df_feat["split_year"] < year
            p90_hist = float(
                target[history_mask].quantile(0.90)
            )

            threshold.loc[current_mask] = p90_hist
            thresholds[str(year)] = p90_hist

    expected_flag = (
        df_feat["lag_revenue_1"] > threshold
    ).astype(int)

    actual_flag = (
        df_feat["is_high_revenue_week"]
        .astype(int)
    )

    current_target_flag = (target > threshold).astype(int)

    return {
        "matches_lag_based_definition": actual_flag.equals(expected_flag),
        "matches_current_target": actual_flag.equals(current_target_flag),
        "n_flagged": int(actual_flag.sum()),
        "thresholds": thresholds,
    }


outlier_rows = []

for name, df_feat in feature_dfs.items():
    result = check_outlier_flag(df_feat)

    outlier_rows.append({
        "series": name,
        "lag_based_definition": result[
            "matches_lag_based_definition"
        ],
        "same_as_current_target": result[
            "matches_current_target"
        ],
        "n_flagged": result["n_flagged"],
    })

    print(f"\n{name.upper()} thresholds:")
    for period, value in result["thresholds"].items():
        print(f"  {period:16s}: {value:.2f}")

outlier_check_df = pd.DataFrame(outlier_rows)
print("\nOutlier leakage check")
display(outlier_check_df)



TOTAL thresholds:
  train_2019_2022 : 4555.32
  2023            : 4555.32
  2024            : 4427.77
  2025            : 4375.16

ONLINE thresholds:
  train_2019_2022 : 1588.10
  2023            : 1588.10
  2024            : 1603.90
  2025            : 1610.57

STORE thresholds:
  train_2019_2022 : 1682.99
  2023            : 1682.99
  2024            : 1688.43
  2025            : 1687.55

MOBILE thresholds:
  train_2019_2022 : 1853.30
  2023            : 1853.30
  2024            : 1784.82
  2025            : 1762.04

Outlier leakage check


,series,lag_based_definition,same_as_current_target,n_flagged
0,total,True,False,33
1,online,True,False,43
2,store,True,False,38
3,mobile,True,False,27


In [43]:
def check_ratio_features(df_feat):

    expected_52 = (df_feat["lag_revenue_1"]/(df_feat["lag_revenue_52"].abs()+ 1e-8))

    expected_4 = (df_feat["lag_revenue_1"]/(df_feat["lag_revenue_4"].abs()+ 1e-8))

    return {
        "lag_52_ratio_ok": np.allclose(
            df_feat["lag_52_ratio"].to_numpy(dtype=float),
            expected_52.to_numpy(dtype=float),
            equal_nan=True,
        ),

        "lag_4_ratio_ok": np.allclose(
            df_feat["lag_4_ratio"].to_numpy(dtype=float),
            expected_4.to_numpy(dtype=float),
            equal_nan=True,
        ),
    }


ratio_rows = []

for name, df_feat in feature_dfs.items():
    result = check_ratio_features(df_feat)
    result["series"] = name
    ratio_rows.append(result)

ratio_check_df = pd.DataFrame(ratio_rows)[
    ["series", "lag_52_ratio_ok", "lag_4_ratio_ok"]
]

display(ratio_check_df)


,series,lag_52_ratio_ok,lag_4_ratio_ok
0,total,True,True
1,online,True,True
2,store,True,True
3,mobile,True,True


In [44]:
def check_split_boundaries(df_feat):
    rows = []

    for year in [2023, 2024, 2025]:

        current_mask = df_feat["split_year"] == year
        history_mask = df_feat["split_year"] < year

        if not current_mask.any() or not history_mask.any():
            continue

        first_idx = df_feat.index[current_mask][0]
        last_hist_idx = df_feat.index[history_mask][-1]

        lag1 = float(df_feat.loc[first_idx, "lag_revenue_1"])
        previous_target = float(df_feat.loc[last_hist_idx, "target_revenue"])


        current_target = float(df_feat.loc[first_idx, "target_revenue"])

        rows.append({
            "year": year,
            "first_week": first_idx,
            "previous_week": last_hist_idx,
            "lag_1": lag1,
            "expected_previous_target": previous_target,
            "current_target": current_target,
            "lag_matches_previous": np.isclose(
                lag1,
                previous_target
            ),
            "lag_equals_current_target": np.isclose(
                lag1,
                current_target
            ),
        })

    return pd.DataFrame(rows)


boundary_summary = []

for name, df_feat in feature_dfs.items():
    result = check_split_boundaries(df_feat)
    boundary_summary.append({
        "series": name,
        "all_boundaries_pass":
            bool(result["lag_matches_previous"].all()),
        "lag_not_current_target":
            bool((~result["lag_equals_current_target"]).all()),
    })

    print(f"\n{name.upper()}")
    display(result)

boundary_summary_df = pd.DataFrame(boundary_summary)

print("\nBoundary summary")
display(boundary_summary_df)



TOTAL


,year,first_week,previous_week,lag_1,expected_previous_target,current_target,lag_matches_previous,lag_equals_current_target
0,2023,2023-01-02,2022-12-26,3004.282507,3004.282507,2949.653609,True,False
1,2024,2024-01-01,2023-12-25,3869.144708,3869.144708,2614.403979,True,False
2,2025,2025-01-06,2024-12-30,3005.906967,3005.906967,3126.140947,True,False



ONLINE


,year,first_week,previous_week,lag_1,expected_previous_target,current_target,lag_matches_previous,lag_equals_current_target
0,2023,2023-01-02,2022-12-26,1349.830512,1349.830512,759.645157,True,False
1,2024,2024-01-01,2023-12-25,1298.917656,1298.917656,968.639586,True,False
2,2025,2025-01-06,2024-12-30,724.518749,724.518749,1230.584321,True,False



STORE


,year,first_week,previous_week,lag_1,expected_previous_target,current_target,lag_matches_previous,lag_equals_current_target
0,2023,2023-01-02,2022-12-26,924.005974,924.005974,991.345456,True,False
1,2024,2024-01-01,2023-12-25,1242.566737,1242.566737,764.625383,True,False
2,2025,2025-01-06,2024-12-30,1429.473769,1429.473769,964.436295,True,False



MOBILE


,year,first_week,previous_week,lag_1,expected_previous_target,current_target,lag_matches_previous,lag_equals_current_target
0,2023,2023-01-02,2022-12-26,730.446020,730.446020,1198.662996,True,False
1,2024,2024-01-01,2023-12-25,1327.660315,1327.660315,881.139010,True,False
2,2025,2025-01-06,2024-12-30,851.914449,851.914449,931.120331,True,False



Boundary summary


,series,all_boundaries_pass,lag_not_current_target
0,total,True,True
1,online,True,True
2,store,True,True
3,mobile,True,True


In [45]:
def check_time_features(df_feat):

    idx = df_feat.index
    woy = idx.isocalendar().week.astype(int)

    expected = {
        "week_of_year": woy,
        "month": pd.Series(idx.month, index=idx),
        "quarter": pd.Series(idx.quarter, index=idx),
        "year": pd.Series(idx.year, index=idx),
        "week_sin": pd.Series(
            np.sin(2 * np.pi * woy.to_numpy() / 52),
            index=idx
        ),
        "week_cos": pd.Series(
            np.cos(2 * np.pi * woy.to_numpy() / 52),
            index=idx
        ),
        "trend_index": pd.Series(
            np.arange(len(df_feat)),
            index=idx
        ),
    }

    result = {}

    for col, expected_values in expected.items():
        if col not in df_feat.columns:
            result[col] = False
            continue

        result[col] = np.allclose(
            df_feat[col].to_numpy(dtype=float),
            np.asarray(expected_values, dtype=float),
            equal_nan=True,
        )

    return result


time_rows = []

for name, df_feat in feature_dfs.items():
    result = check_time_features(df_feat)
    result["series"] = name
    time_rows.append(result)

time_check_df = pd.DataFrame(time_rows)

display(time_check_df)


,week_of_year,month,quarter,year,week_sin,week_cos,trend_index,series
0,True,True,True,True,True,True,True,total
1,True,True,True,True,True,True,True,online
2,True,True,True,True,True,True,True,store
3,True,True,True,True,True,True,True,mobile


In [46]:
nan_tables = {}
for name, df_feat in feature_dfs.items():
    feature_cols = [
        c for c in df_feat.columns
        if c not in [
            "target_revenue",
            "split_year"
        ]
    ]

    nan_counts = (df_feat[feature_cols].isna().sum().sort_values(ascending=False))
    nan_counts = nan_counts[nan_counts > 0]
    nan_tables[name] = nan_counts

    print(f"\n{name.upper()}")
    display(nan_counts.to_frame("NaN count"))



TOTAL


,NaN count
lag_52_ratio,52
lag_revenue_52,52
lag_revenue_26,26
rolling_min_26,26
rolling_max_26,26
rolling_mean_26,26
rolling_std_26,26
lag_revenue_12,12
rolling_max_12,12
rolling_std_12,12



ONLINE


,NaN count
lag_52_ratio,52
lag_revenue_52,52
lag_revenue_26,26
rolling_min_26,26
rolling_max_26,26
rolling_mean_26,26
rolling_std_26,26
lag_revenue_12,12
rolling_max_12,12
rolling_std_12,12



STORE


,NaN count
lag_52_ratio,52
lag_revenue_52,52
lag_revenue_26,26
rolling_min_26,26
rolling_max_26,26
rolling_mean_26,26
rolling_std_26,26
lag_revenue_12,12
rolling_max_12,12
rolling_std_12,12



MOBILE


,NaN count
lag_52_ratio,52
lag_revenue_52,52
lag_revenue_26,26
rolling_min_26,26
rolling_max_26,26
rolling_mean_26,26
rolling_std_26,26
lag_revenue_12,12
rolling_max_12,12
rolling_std_12,12


In [47]:
summary_rows = []

for name, df_feat in feature_dfs.items():

    lag_result = check_lag_features(df_feat)
    rolling_result = check_rolling_features(df_feat)
    outlier_result = check_outlier_flag(df_feat)
    ratio_result = check_ratio_features(df_feat)
    boundary_result = check_split_boundaries(df_feat)
    time_result = check_time_features(df_feat)

    lag_ok = bool(lag_result["passed"].all())

    rolling_ok = bool(rolling_result["passed"].all())

    outlier_ok = bool(outlier_result["matches_lag_based_definition"])

    outlier_no_current_target = not bool(outlier_result["matches_current_target"])

    ratio_ok = bool(
        ratio_result["lag_52_ratio_ok"]
        and ratio_result["lag_4_ratio_ok"]
    )

    boundary_ok = bool(
        boundary_result["lag_matches_previous"].all()
    )

    time_ok = bool(
        all(time_result.values())
    )

    all_pass = all([
        lag_ok,
        rolling_ok,
        outlier_ok,
        outlier_no_current_target,
        ratio_ok,
        boundary_ok,
        time_ok,
    ])

    summary_rows.append({
        "series": name,
        "lag": lag_ok,
        "rolling": rolling_ok,
        "outlier_lag_based": outlier_ok,
        "outlier_not_current_target":
            outlier_no_current_target,
        "ratio": ratio_ok,
        "temporal_boundary": boundary_ok,
        "time_features": time_ok,
        "ALL_PASS": all_pass,
    })


feature_leakage_summary = pd.DataFrame(summary_rows)

display(feature_leakage_summary)

if feature_leakage_summary["ALL_PASS"].all():
    print("PASSED")
else:
    print("NOT PASSED")


,series,lag,rolling,outlier_lag_based,outlier_not_current_target,ratio,temporal_boundary,time_features,ALL_PASS
0,total,True,True,True,True,True,True,True,True
1,online,True,True,True,True,True,True,True,True
2,store,True,True,True,True,True,True,True,True
3,mobile,True,True,True,True,True,True,True,True


PASSED


## Kết luận
- Các tập train, validation và test không bị chồng lấn về `week_start`; ranh giới thời gian giữa các giai đoạn được tách biệt đúng thứ tự.
- Chuỗi thời gian được kiểm tra về thứ tự, tính liên tục, tuần bị thiếu và tuần trùng lặp; không phát hiện bất thường nghiêm trọng trong các kiểm tra tương ứng.
- Biến mục tiêu `revenue` không có missing value.
- Các lag feature khớp với `target_revenue.shift(k)` và các rolling feature chỉ sử dụng dữ liệu từ quá khứ thông qua `shift(1)` --> vì vậy không sử dụng doanh thu của tuần hiện tại.
- Các ratio feature được tính từ các biến lag, không lấy trực tiếp giá trị mục tiêu hiện tại.
- Cờ `is_high_revenue_week` được xác định dựa trên `lag_revenue_1` và ngưỡng p90 của dữ liệu lịch sử theo từng giai đoạn; không trùng với cách xác định dựa trên target hiện tại.
- Các kiểm tra tại ranh giới năm 2023, 2024 và 2025 cho thấy `lag_1` của tuần đầu mỗi giai đoạn khớp với target của tuần liền trước, không phải target của chính tuần đó.
- Các biến thời gian như tuần trong năm, tháng, quý, năm, biến sin/cos mùa vụ và `trend_index` được tái tạo đúng từ `week_start`.
- `target_revenue` và `split_year` được loại khỏi tập biến đầu vào `X`, nên không bị đưa trực tiếp vào feature set.

==> Các feature hiện tại phù hợp với nguyên tắc dự báo theo thời gian và có thể được sử dụng cho bước huấn luyện mô hình.